#Introduction


 L’objectif de ce projet est de concevoir un moteur de recherche intelligent basé sur le traitement du langage naturel (NLP), capable de comprendre des requêtes plus complexes et plus proches du langage humain. Contrairement aux moteurs traditionnels, ce système permet à l’utilisateur de rechercher des chansons à partir d’idées, d’émotions ou de fragments de paroles, même approximatifs.
Plus concrètement, au lieu de se limiter à une recherche par mots-clés, le système transforme chaque texte de paroles en une représentation numérique appelée embedding. Cette transformation permet de capturer le sens des phrases plutôt que leur simple forme.


[Lien du dataset](https://www.kaggle.com/datasets/imuhammad/audio-features-and-lyrics-of-spotify-songs)

In [2]:
import pandas as pd
import re
import kagglehub
from sentence_transformers import SentenceTransformer
import numpy as np
!pip install faiss-cpu
import faiss

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 58.8 MB/s eta 0:00:00


In [3]:
path = kagglehub.dataset_download("imuhammad/audio-features-and-lyrics-of-spotify-songs")
data = pd.read_csv(f"{path}/spotify_songs.csv")

100%|██████████| 12.3M/12.3M [00:00<00:00, 52.1MB/s]

Extracting files...


In [4]:
data.head()

,track_id,track_name,track_artist,lyrics,track_popularity,track_album_id,track_album_name,track_album_release_date,playlist_name,playlist_id,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,language
0,0017A6SJgTbfQVU2EtsPNo,Pangarap,Barbie's Cradle,Minsan pa Nang ako'y napalingon Hindi ko alam ...,41,1srJQ0njEQgd8w4XSqI4JQ,Trip,2001-01-01,Pinoy Classic Rock,37i9dQZF1DWYDQ8wBxd7xt,...,-10.068,1,0.0236,0.27900,0.01170,0.0887,0.566,97.091,235440,tl
1,004s3t0ONYlzxII9PLgU6z,I Feel Alive,Steady Rollin,"The trees, are singing in the wind The sky blu...",28,3z04Lb9Dsilqw68SHt6jLB,Love & Loss,2017-11-21,Hard Rock Workout,3YouF0u7waJnolytf9JCXf,...,-4.739,1,0.0442,0.01170,0.00994,0.3470,0.404,135.225,373512,en
2,00chLpzhgVjxs1zKC9UScL,Poison,Bell Biv DeVoe,"NA Yeah, Spyderman and Freeze in full effect U...",0,6oZ6brjB8x3GoeSYdwJdPc,Gold,2005-01-01,"Back in the day - R&B, New Jack Swing, Swingbe...",3a9y4eeCJRmG9p4YKfqYIx,...,-7.504,0,0.2160,0.00432,0.00723,0.4890,0.650,111.904,262467,en
3,00cqd6ZsSkLZqGMlQCR0Zo,Baby It's Cold Outside (feat. Christina Aguilera),CeeLo Green,I really can't stay Baby it's cold outside I'v...,41,3ssspRe42CXkhPxdc12xcp,CeeLo's Magic Moment,2012-10-29,Christmas Soul,6FZYc2BvF7tColxO8PBShV,...,-5.819,0,0.0341,0.68900,0.00000,0.0664,0.405,118.593,243067,en
4,00emjlCv9azBN0fzuuyLqy,Dumb Litty,KARD,Get up out of my business You don't keep me fr...,65,7h5X3xhh3peIK9Y0qI5hbK,KARD 2nd Digital Single ‘Dumb Litty’,2019-09-22,K-Party Dance Mix,37i9dQZF1DX4RDXswvP6Mj,...,-1.993,1,0.0409,0.03700,0.00000,0.1380,0.240,130.018,193160,en


In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18454 entries, 0 to 18453
Data columns (total 25 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   track_id                  18454 non-null  object 
 1   track_name                18454 non-null  object 
 2   track_artist              18454 non-null  object 
 3   lyrics                    18194 non-null  object 
 4   track_popularity          18454 non-null  int64  
 5   track_album_id            18454 non-null  object 
 6   track_album_name          18454 non-null  object 
 7   track_album_release_date  18454 non-null  object 
 8   playlist_name             18454 non-null  object 
 9   playlist_id               18454 non-null  object 
 10  playlist_genre            18454 non-null  object 
 11  playlist_subgenre         18454 non-null  object 
 12  danceability              18454 non-null  float64
 13  energy                    18454 non-null  float64
 14  key   

In [6]:
data.describe(include="all")

,track_id,track_name,track_artist,lyrics,track_popularity,track_album_id,track_album_name,track_album_release_date,playlist_name,playlist_id,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,language
count,18454,18454,18454,18194,18454.000000,18454,18454,18454,18454,18454,...,18454.000000,18454.000000,18454.000000,18454.000000,18454.000000,18454.000000,18454.000000,18454.000000,18454.000000,18194
unique,18454,15198,6031,15977,NaN,14278,12671,3817,442,464,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,34
top,7zzZmpw8L66ZPjH1M6qmOs,Poison,Queen,Lyrics for this song have yet to be released. ...,NaN,6TBwXfQCeLoVIOW53dNLqz,Greatest Hits,2013-01-01,Indie Poptimism,3E88dLx4fgFYY70gdGzdnB,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,en
freq,1,13,125,48,NaN,16,108,165,258,94,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15405
mean,NaN,NaN,NaN,NaN,42.438821,NaN,NaN,NaN,NaN,NaN,...,-6.769159,0.580525,0.106192,0.175348,0.051216,0.189593,0.520598,120.812167,230319.306763,NaN
std,NaN,NaN,NaN,NaN,24.616740,NaN,NaN,NaN,NaN,NaN,...,2.920757,0.493487,0.102291,0.217795,0.168263,0.153751,0.228716,27.586424,57255.086685,NaN
min,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,...,-34.283000,0.000000,0.022400,0.000001,0.000000,0.009360,0.000010,37.114000,31893.000000,NaN
25%,NaN,NaN,NaN,NaN,24.000000,NaN,NaN,NaN,NaN,NaN,...,-8.249000,0.000000,0.039700,0.016100,0.000000,0.092700,0.345000,98.856000,193230.250000,NaN
50%,NaN,NaN,NaN,NaN,48.000000,NaN,NaN,NaN,NaN,NaN,...,-6.227000,1.000000,0.060300,0.081900,0.000009,0.128000,0.522000,120.045000,221340.000000,NaN
75%,NaN,NaN,NaN,NaN,62.000000,NaN,NaN,NaN,NaN,NaN,...,-4.719000,1.000000,0.130000,0.254000,0.001720,0.246000,0.700000,135.984000,258078.250000,NaN


#Nettoyage


Dans le cadre de la préparation des données, une étape essentielle consiste à analyser la qualité des paroles présentes dans la variable lyrics. En effet, bien que le dataset contienne un grand nombre de chansons, toutes les entrées ne sont pas exploitables pour un traitement en NLP. Certaines lignes contiennent des valeurs manquantes, des textes très courts ou encore des contenus peu informatifs comme “nan”, “NA NA”, “instrumental” ou de simples expressions répétitives.
Ce traitement parcourt l’ensemble des paroles et sélectionne uniquement celles dont la longueur est inférieure à 100 caractères. L’objectif n’est pas encore de les supprimer directement, mais plutôt de mettre en évidence les entrées potentiellement inutiles ou de mauvaise qualité. Les résultats obtenus montrent clairement la présence de nombreux textes non exploitables, tels que des valeurs nulles transformées en chaîne de caractères (“nan”), des mentions d’instrumental, ou encore des phrases trop courtes pour porter un sens réel.

In [7]:
bad_values = [str(i) for i in data['lyrics'] if len(str(i)) < 100]
bad_values

['Down, down do-down, now ge-get',
 'nan',
 'nan',
 'nan',
 "I'm losing it I'm losing it I'm losing it I'm losing it I'm losing it I'm losing it I'm losing it",
 'nan',
 '(instrumental)',
 'nan',
 'You have five seconds to terminate this tape Five Four Three Two One',
 'nan',
 'NA NA',
 "Everybody that's in the place Let's go!",
 'Instrumental',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'Everybody know Marsh- Woah Everybody know Marsh- Woah',
 'nan',
 'nan',
 'Play nice',
 "Hallelujah, oh, I'm down on the beach Hallelujah, oh, I'm down on the beach",
 'nan',
 'PUT YOUR HANDS UP! PUT YOUR HANDS UP!',
 'nan',
 "We're the fuckin' animals We're the fuckin' animals",
 'Bird, Bird Bird, Bird, Bird, Bird, Bird Bird, Bird Bird, Bird, Bird, Bird, Bird Machine check',
 'nan',
 'nan',
 "Don't Lyrics",
 'Letra de "Calientito Boyz"',
 'nan',
 'nan',
 'nan',
 'Lyrics for this song have yet to be released. Please check back once the song has been released.',
 'nan',
 'nan',
 'nan',
 'nan',
 'G

suppression des valeurs manquant


In [8]:
data = data.dropna(subset=['lyrics'])
data['lyrics'] = data['lyrics'].astype(str).str.lower().str.strip()

data = data[~data['lyrics'].isin(bad_values)]

# Normalisation

 Rendre les textes en minuscule et suppression des caractères spéciaux qui pourrait polluer les données

In [9]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text

In [10]:
cat_keys = ["track_id",
 "track_name",
 "track_artist",
 "lyrics",
 "track_album_id",
 "track_album_name",
 "track_album_release_date",
 "playlist_name",
 "playlist_id",
 "playlist_genre",
 "playlist_subgenre"
]

for i in cat_keys:
    data[i] = data[i].apply(clean_text)


In [11]:
data.keys()

Index(['track_id', 'track_name', 'track_artist', 'lyrics', 'track_popularity',
       'track_album_id', 'track_album_name', 'track_album_release_date',
       'playlist_name', 'playlist_id', 'playlist_genre', 'playlist_subgenre',
       'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness',
       'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo',
       'duration_ms', 'language'],
      dtype='object')

In [12]:
data['combined_text'] = (
    "Artist: " + data['track_artist'].fillna('Unknown').astype(str) +
    " , Title: " + data['track_name'].fillna('Unknown').astype(str) +
    " , Album: " + data['track_album_name'].fillna('Unknown').astype(str) +
    " , Lyrics: " + data['lyrics'].fillna('').astype(str)
)

In [13]:
data['combined_text']

,combined_text
0,"Artist: barbie's cradle , Title: pangarap , Al..."
1,"Artist: steady rollin , Title: i feel alive , ..."
2,"Artist: bell biv devoe , Title: poison , Album..."
3,"Artist: ceelo green , Title: baby it's cold ou..."
4,"Artist: kard , Title: dumb litty , Album: kard..."
...,...
18449,"Artist: qulinez , Title: rising like the sun r..."
18450,"Artist: nicki minaj , Title: anaconda , Album:..."
18451,"Artist: ponderosa twins plus one , Title: boun..."
18452,"Artist: father mc , Title: i'll do 4 u re reco..."


sauvegarde de donné nettoyé

In [18]:
data.to_csv('lyrx_cleaned.csv', index=False)

# Embedding


initialiation du model

In [14]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
sentences = data['combined_text'].tolist()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

encodage

In [ ]:
embeddings = model.encode(sentences, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
np.save('music_embeddings.npy', embeddings)

In [15]:
embeddings = np.load('music_embeddings.npy').astype('float32')
dimension = embeddings.shape[1]

# Créer l'index. "IndexFlatIP" calcule le produit scalaire (Inner Product)
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print(f"Index prêt avec {index.ntotal} chansons.")

Index prêt avec 18192 chansons.


In [16]:
def search_engine(user_query, k=10):
    query_vector = model.encode([user_query]).astype('float32')

    distances, indices = index.search(query_vector, k)

    results = data.iloc[indices[0]].copy()
    results['score'] = distances[0]

    return results[['track_name', 'track_artist', 'playlist_genre', 'score']]

In [26]:
search_engine("ed sheeran perfect")

,track_name,track_artist,playlist_genre,score
1871,perfect,ed sheeran,latin,2.173651
2846,perfect duet ed sheeran beyonc,ed sheeran,r b,2.116525
4392,happier ti sto s aftr hrs remix,ed sheeran,pop,2.069699
12229,perfect mike perry remix,ed sheeran,edm,2.061550
9217,smooth operator single version,sade,r b,2.029345
11209,smooth operator remastered,sade,rock,2.011662
411,fortunate,maxwell,r b,1.994796
14179,well done,sean c johnson,r b,1.986678
15494,the logical song remastered 2010,supertramp,rock,1.939516
5188,miracle in the middle of my heart radio edit,cl ment bcx,pop,1.866093
